In [ ]:
import pickle

import matplotlib
import matplotlib.pyplot as plt
import os
import mne
import numpy as np
import pandas as pd
import torch
import copy

from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import matplotlib.pylab as pylab
import gc
import quantus
from tqdm import tqdm
from captum.attr import GradientShap

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""


def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))

In [ ]:
def load_data_set(subject_index=2):
    cfg = load_config()
    cfg.dataset.subject_index =  subject_index
    all_epochs, labels_raw, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    print(all_epochs.shape)
    
    return all_epochs[-200:,:,:900], ch_names

In [ ]:
def load_model(cfg, start_index=100, subject_index=2):
    save_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/model_checkpoints/finetune"
    
    file_path = os.path.join(save_path, f"subject_{subject_index}", f"model_checkpoint_finetune_subject_index_{subject_index}_start_idx_{start_index}_rep_0_pen.pth") 
    trunk_net = TrunkNet(n_chans=input_shape_st[0], n_times=input_shape_st[1])
    head_net = HeadNet(64, 1)  # Assuming these are the correct dimensions
    model = S4PatchedFinalNet(64, trunk_net, head_net)
    
    weights = torch.load(file_path)
    model.load_state_dict(weights)
    #model.load_state_dict(full_checkpoint['model_state_dict'])
    model.eval()
    model.to(device)
    return model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_shape_st = (60,900)

In [ ]:
def compute_predictions_RCAV(subject_index):
    with torch.no_grad():
        input_shape_st = (60, 900)
        cfg = load_config()
        cfg.dataset.subject_index = subject_index
        batch_size = cfg.training.batch_size
        all_predictions = {}
        all_epochs,_= load_data_set(subject_index=subject_index)
        for model_index in [100,200,300,400,500]:
            with torch.no_grad():
                model = load_model(cfg, start_index=model_index, subject_index=subject_index)
                inputs = torch.from_numpy(all_epochs)
                inputs = inputs.to(device).float()
                    
                pred_mean = model(inputs)[:, 0]
        
                pred_label= pred_mean.cpu().numpy()
        
                all_predictions[model_index] = pred_label
         

   
        np.save(f"RCAV_predictions_subject_{subject_index}.npy", all_predictions)



In [ ]:
compute_predictions_RCAV(subject_index=2)

In [ ]:
preds = np.load("/home/marco/Documents/GitHub/tms_eeg_decoding/RCAV/RCAV_predictions_subject_2.npy", allow_pickle=True).item()

In [ ]:
preds

In [ ]:
_, ch_names = load_data_set(subject_index=2)

In [ ]:
freq_bands = {
              "delta": (0, 4),
              "theta": (0, 4),             
              "alpha": (8, 12),
              "beta": (12, 30),
              "gamma": (30, 45)}

amplification_factors = [0.5,0.8,0.9,1.1,1.2,1.5]

In [ ]:
def load_distances(subject_index, rep=1):
    dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/RCAV/RCAV_parallel_perturbation_distance_new"
    
    distances = {}

    for band_name, (low_freq, high_freq) in freq_bands.items():
        distances[band_name] = {}

        for factor in amplification_factors:
            file_path = f"RCAV_parallel_distance_dict_{band_name}_channel_amp_factor_{factor}_subject_{subject_index}_rep_1.npy"
            load_path = os.path.join(dir, file_path)
            distances[band_name][factor] = np.load(load_path, allow_pickle=True).item()

    return distances

In [ ]:
pert_preds = np.load("/home/marco/Documents/GitHub/tms_eeg_decoding/RCAV/RCAV_parallel_perturbation_new/RCAV_parallel_perturbed_prediction_dict_alpha_channel_amp_factor_1.5_subject_2_rep_1.npy", allow_pickle=True).item()

In [ ]:
pert_dist = np.load("/home/marco/Documents/GitHub/tms_eeg_decoding/RCAV/RCAV_parallel_perturbation_distance_new/RCAV_parallel_distance_dict_alpha_channel_amp_factor_0.5_subject_2_rep_1.npy", allow_pickle=True).item()

In [ ]:

def calculate_diff_per_channel_distance_weighted(pred_label_original, freq_bands, amplification_factors, ch_names, distances, subject_index=2, rep=1, mi=100):
    dir_constrained = "/home/marco/Documents/GitHub/tms_eeg_decoding/RCAV/RCAV_parallel_perturbation_new"
    mean_diff_per_channel = {}
    median_diff_per_channel = {}

    for band_name, (low_freq, high_freq) in freq_bands.items():
        mean_diff_per_channel[band_name] = {}
        median_diff_per_channel[band_name] = {}
        for factor in amplification_factors:
            file_path = f"RCAV_parallel_perturbed_prediction_dict_{band_name}_channel_amp_factor_{factor}_subject_{subject_index}_rep_1.npy"
            load_path = os.path.join(dir_constrained, file_path)
            perturbed_data = np.load(load_path, allow_pickle=True).item()
            mean_diff_per_channel[band_name][factor] = {}
            median_diff_per_channel[band_name][factor] = {}
            for ch_name in ch_names:
                perturbed_amplitude = np.array(perturbed_data[mi][ch_name])
                diff = np.abs(pred_label_original - perturbed_amplitude)/np.array(distances[band_name][factor][mi][ch_name])
                mean_diff_per_channel[band_name][factor][ch_name] = np.mean(diff)
                median_diff_per_channel[band_name][factor][ch_name] = np.median(diff)
    
    return mean_diff_per_channel, median_diff_per_channel

In [ ]:
distances = load_distances(subject_index=2, rep=1)

In [ ]:
all_median_diff = {}
all_mean_diff = {}
for mi in [100,200,300,400,500]:
    all_mean_diff[mi], all_median_diff[mi] = calculate_diff_per_channel_distance_weighted(preds[mi], freq_bands, amplification_factors, ch_names, distances, subject_index=2, rep=1, mi=mi)


In [ ]:
np.save("/home/marco/Documents/GitHub/tms_eeg_decoding/RCAV/RCAV_parallel_perturbation_new/mean_diff_per_channel.npy", all_mean_diff)
np.save("/home/marco/Documents/GitHub/tms_eeg_decoding/RCAV/RCAV_parallel_perturbation_new/median_diff_per_channel.npy", all_median_diff)